In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import os
import warnings
warnings.filterwarnings('ignore')

DATA = r'C:\Users\shahb\Desktop\Research\Datasets'
MIMIC = DATA + r'\mimic-iv-3.1'
MOVER = DATA + r'\MOVER'

pd.set_option('display.max_columns', 30)

In [ ]:
# load admissions and procedures
adm = pd.read_csv(MIMIC + r'\hosp\admissions.csv.gz')
proc = pd.read_csv(MIMIC + r'\hosp\procedures_icd.csv.gz')
pts = pd.read_csv(MIMIC + r'\hosp\patients.csv.gz')
dx = pd.read_csv(MIMIC + r'\hosp\diagnoses_icd.csv.gz')

print(adm.shape, proc.shape)
print(adm.columns.tolist())

In [ ]:
# surgical admissions only
surg_hadm = proc['hadm_id'].unique()
surg_adm = adm[adm['hadm_id'].isin(surg_hadm)].copy()
print('surgical admissions:', len(surg_adm))

In [ ]:
# cardiac outcome - acute events only (MI + arrest)
# tried broad def first (afib, chf etc) but event rate was 26% which is way too high
# stuck with acute cardiac events only
acute_codes = ['I21', 'I22', 'I46', '4100', '4101', '4102', '4103',
               '4104', '4105', '4106', '4107', '4108', '4109', '4275']
acute_mask = dx['icd_code'].str.startswith(tuple(acute_codes), na=False)
acute_hadm = dx[acute_mask]['hadm_id'].unique()

surg_adm['CARDIAC_OUTCOME'] = surg_adm['hadm_id'].isin(acute_hadm).astype(int)
print('event rate:', surg_adm['CARDIAC_OUTCOME'].mean().round(4))
print('events:', surg_adm['CARDIAC_OUTCOME'].sum())

In [ ]:
# icu vitals from chartevents - big file, need chunks
vital_items = [220045, 220052, 220181, 220179, 220180]

chunks = []
for chunk in pd.read_csv(
    MIMIC + r'\icu\chartevents.csv.gz',
    chunksize=1000000,
    usecols=['hadm_id', 'itemid', 'charttime', 'valuenum']):
    chunk = chunk[
        (chunk['hadm_id'].isin(surg_hadm)) &
        (chunk['itemid'].isin(vital_items)) &
        (chunk['valuenum'].notna())]
    if len(chunk) > 0:
        chunks.append(chunk)

vitals = pd.concat(chunks, ignore_index=True)
print(vitals.shape)
print(vitals['itemid'].value_counts())

In [ ]:
item_names = {
    220045: 'HR', 220052: 'MAP_art',
    220181: 'MAP_noninv', 220179: 'BP_sys', 220180: 'BP_dia'
}
vitals['meas'] = vitals['itemid'].map(item_names)

# hr features
hr = vitals[vitals['meas'] == 'HR']
hr_feats = hr.groupby('hadm_id')['valuenum'].agg(
    HR_mean='mean', HR_min='min', HR_max='max', HR_std='std').reset_index()

# map
mapdf = vitals[vitals['meas'].isin(['MAP_art', 'MAP_noninv'])].copy()
map_feats = mapdf.groupby('hadm_id')['valuenum'].agg(
    MAP_mean='mean', MAP_min='min', MAP_std='std').reset_index()
mapdf['hypo'] = (mapdf['valuenum'] < 65).astype(int)
map_hypo = mapdf.groupby('hadm_id')['hypo'].mean().reset_index().rename(
    columns={'hypo': 'MAP_pct_below65'})
map_feats = map_feats.merge(map_hypo, on='hadm_id')

intraop = hr_feats.merge(map_feats, on='hadm_id', how='outer')
print(intraop.shape)

In [ ]:
# comorbidities
def flag_hadm(codes):
    m = dx['icd_code'].str.startswith(tuple(codes), na=False)
    return set(dx[m]['hadm_id'])

htn  = flag_hadm(['401', 'I10', 'I11', 'I12', 'I13'])
dm   = flag_hadm(['250', 'E11', 'E10', 'E12', 'E13'])
chf  = flag_hadm(['428', 'I50'])
cad  = flag_hadm(['414', 'I25'])
copd = flag_hadm(['496', 'J44'])
ckd  = flag_hadm(['585', 'N18'])
mi   = flag_hadm(['410', 'I21', 'I22'])
afib = flag_hadm(['427.31', 'I48'])
strk = flag_hadm(['434', 'I63', 'I64'])
pvd  = flag_hadm(['443', 'I73'])

In [ ]:
# build analytic dataset
df = surg_adm.merge(intraop, on='hadm_id', how='left')
df = df.merge(pts[['subject_id', 'anchor_age']], on='subject_id', how='left')

df['hypertension'] = df['hadm_id'].isin(htn).astype(int)
df['diabetes']     = df['hadm_id'].isin(dm).astype(int)
df['chf']          = df['hadm_id'].isin(chf).astype(int)
df['cad']          = df['hadm_id'].isin(cad).astype(int)
df['copd']         = df['hadm_id'].isin(copd).astype(int)
df['ckd']          = df['hadm_id'].isin(ckd).astype(int)
df['prior_mi']     = df['hadm_id'].isin(mi).astype(int)
df['afib']         = df['hadm_id'].isin(afib).astype(int)
df['stroke']       = df['hadm_id'].isin(strk).astype(int)
df['pvd']          = df['hadm_id'].isin(pvd).astype(int)

print(df.shape, df['CARDIAC_OUTCOME'].sum())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

feats = ['anchor_age', 'hypertension', 'diabetes', 'chf', 'cad', 'copd',
         'ckd', 'prior_mi', 'afib', 'stroke', 'pvd',
         'HR_mean', 'HR_min', 'HR_max', 'HR_std',
         'MAP_mean', 'MAP_min', 'MAP_std', 'MAP_pct_below65']

# drop rows missing HR - need intraop data
df2 = df.dropna(subset=['HR_mean']).copy()
X = df2[feats]
y = df2['CARDIAC_OUTCOME']
print(f'{len(df2)} cases, {y.sum()} events ({y.mean()*100:.1f}%)')

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.2, 
                                        random_state=42, stratify=y)

spw = len(ytr)/ytr.sum()
xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=.05,
                    scale_pos_weight=spw, random_state=42, eval_metric='auc')
xgb.fit(Xtr, ytr)
proba = xgb.predict_proba(Xte)[:,1]
print('MIMIC AUC:', roc_auc_score(yte, proba).round(3))

In [ ]:
# shap - compare with mover
explainer = shap.TreeExplainer(xgb)
shap_vals = explainer.shap_values(Xte)

shap.summary_plot(shap_vals, Xte, plot_type='bar', show=False, max_display=15)
plt.title('SHAP feature importance — MIMIC-IV (BIDMC)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(MOVER + r'\figures\figure6_shap_beeswarm_mimic.png', dpi=300)
plt.show()

In [ ]:
# cross dataset comparison figure
# load mover shap values - need to rerun mover model first
# see periop_cardiac_analysis.ipynb for mover model code
# this just does the side by side figure

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

plt.sca(axes[0])
# mover shap - run periop_cardiac_analysis.ipynb first to get shap_vals_mover
# shap.summary_plot(shap_vals_mover, X2_te, plot_type='bar', show=False, max_display=15)
axes[0].set_title('MOVER (UCI)\nAUC=0.823, event rate=0.77%',
                   fontsize=13, fontweight='bold')

plt.sca(axes[1])
shap.summary_plot(shap_vals, Xte, plot_type='bar', show=False, max_display=15)
axes[1].set_title('MIMIC-IV (BIDMC)\nAUC=0.960, event rate=13.22%',
                   fontsize=13, fontweight='bold')

plt.suptitle('Cross-dataset feature importance', fontsize=15, fontweight='bold')
plt.subplots_adjust(top=.90, bottom=.10, wspace=.5)
plt.savefig(MOVER + r'\figures\figure5_shap_comparison.png', dpi=300, bbox_inches='tight')
plt.show()